# GloVe — Global Vectors for Word Representation (2014)

_Papers · Sequence & NLP_

**Learn word vectors by fitting their dot products to the log of how often words co-occur across the whole corpus.**

---

This notebook is a practice scaffold for the **GloVe — Global Vectors for Word Representation (2014)** lesson. We build GloVe one piece at a time: first the weighted least-squares term it minimizes, then the global co-occurrence matrix, then the Adagrad fit, and finally the embeddings themselves. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Score one term of the objective J

GloVe's loss (Eq. 8) is a sum over word pairs of a **weighted squared error**. For a single pair it is `f(X_ij) · (w_i·w_j + b_i + b_j − log X_ij)²`: the dot product of two word vectors plus two bias terms should reproduce `log X_ij`, the log co-occurrence count.

The weighting `f` (Eq. 9) down-weights rare pairs and caps the influence of very frequent ones. Here we plug in concrete numbers and print every piece so the residual and the term are easy to trace.

In [ ]:
import numpy as np
import math

# Eq. 9 weighting function: ramps up as (x/xmax)^alpha, then saturates at 1.0
def f(x, xmax, alpha):
    if x < xmax:
        return (x / xmax) ** alpha
    return 1.0

# A single worked pair: two 3-d word vectors, their biases, and one count
wi = np.array([0.5, -0.2, 0.1])
wj = np.array([0.3, 0.4, -0.1])
bi = 0.2
bj = 0.1
X_ij = 30.0

weight = f(X_ij, 100.0, 0.75)          # f(30) with xmax=100, alpha=0.75
dot = wi @ wj                           # w_i . w_j
log_count = math.log(X_ij)              # the target the model fits
residual = dot + bi + bj - log_count    # model prediction minus target (Eq. 7)
term = weight * residual ** 2           # one weighted squared-error term of J (Eq. 8)

print("f(30)      :", round(weight, 6))     # 0.40536
print("dot        :", round(dot, 6))        # 0.06
print("log X      :", round(log_count, 6))  # 3.401197
print("residual   :", round(residual, 6))   # -3.041197
print("term of J  :", round(term, 6))       # 3.749127

### Step 2 — Build the global co-occurrence matrix X

GloVe's defining idea is that it trains on **one global matrix**, not on individual context windows like word2vec. We make a tiny two-topic corpus — royalty/people words and animal words — then count how often each pair of words appears together.

Each co-occurrence is weighted by `1/distance`, so neighbours count more than far-apart words. The result is the symmetric matrix `X` that the fit will try to reproduce.

In [ ]:
# A tiny two-topic corpus: 40 short royalty/people sentences + 30 animal sentences
roy = ['king', 'queen', 'prince', 'princess', 'man', 'woman', 'boy', 'girl']
ani = ['cat', 'dog', 'kitten', 'puppy']

rng = np.random.default_rng(42)
sents = [list(rng.choice(roy, size=5, replace=True)) for _ in range(40)]
sents += [list(rng.choice(ani, size=4, replace=True)) for _ in range(30)]

# Build the vocabulary and a word -> index lookup
vocab = sorted(set(w for s in sents for w in s))
stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# Accumulate ONE global co-occurrence matrix (not per-window training)
X = np.zeros((V, V))
for s in sents:
    for a in range(len(s)):
        for b in range(len(s)):
            if a != b:
                X[stoi[s[a]], stoi[s[b]]] += 1.0 / abs(a - b)   # 1/distance weighting

print("V =", V, " nonzero entries =", int((X > 0).sum()))

### Step 3 — Minimize J with Adagrad

Now we fit the vectors. Each word gets two vectors (`W` and `Wt`) and two biases; GloVe trains them by gradient descent on J, visiting only the **nonzero** cells of `X`.

The optimizer is **Adagrad**: per-parameter learning rates that shrink as a running sum of squared gradients grows, which suits the very different update frequencies of common vs. rare words. We loop 250 epochs and print the final loss.

In [ ]:
D, xmax, alpha, lr = 16, 30.0, 0.75, 0.05
r = np.random.default_rng(7)

# Two vector sets + two bias vectors (GloVe's symmetric main / context split)
W = r.standard_normal((V, D)) * 0.1
Wt = r.standard_normal((V, D)) * 0.1
b = np.zeros(V)
bt = np.zeros(V)

# Adagrad accumulators (start at 1 so the first step is well-scaled)
gW = np.ones_like(W)
gWt = np.ones_like(Wt)
gb = np.ones_like(b)
gbt = np.ones_like(bt)

# Only the populated cells of X contribute to the loss
nz = np.argwhere(X > 0)

def fw(x):
    return np.where(x < xmax, (x / xmax) ** alpha, 1.0)

for epoch in range(250):
    r.shuffle(nz)
    J = 0.0
    for i, j in nz:
        xij = X[i, j]
        res = W[i] @ Wt[j] + b[i] + bt[j] - np.log(xij)   # Eq. 7 residual
        wt = fw(xij)                                       # Eq. 9 weight
        J += wt * res ** 2                                 # Eq. 8 term

        g = 2 * wt * res                                   # dJ/d(residual)
        gwi = g * Wt[j]                                    # gradient wrt W[i]
        gwj = g * W[i]                                     # gradient wrt Wt[j]

        # Adagrad updates: step by lr / sqrt(accumulated squared grad)
        W[i] -= lr / np.sqrt(gW[i]) * gwi
        gW[i] += gwi ** 2
        Wt[j] -= lr / np.sqrt(gWt[j]) * gwj
        gWt[j] += gwj ** 2
        b[i] -= lr / np.sqrt(gb[i]) * g
        gb[i] += g ** 2
        bt[j] -= lr / np.sqrt(gbt[j]) * g
        gbt[j] += g ** 2

print("final J    :", round(J, 4))    # ~0.0016

### Step 4 — Read out the embeddings: neighbours and analogy

GloVe's final embedding for a word is the **sum** `w + w_tilde` of its two vectors. We normalize them and use cosine similarity to find nearest neighbours.

The famous test is vector arithmetic: `king − man + woman` should land near `queen`. We compute that direction, exclude the three input words, and print the top matches — clusters by topic emerge even though the model was never told the topics.

In [ ]:
# GloVe's output embedding is the sum of the two vector sets
E = W + Wt
En = E / np.linalg.norm(E, axis=1, keepdims=True)   # unit vectors for cosine similarity
sim = En @ En.T                                       # full word-word cosine matrix

def neighbors(word, k=3):
    i = stoi[word]
    s = sim[i].copy()
    s[i] = -9                                          # never return the word itself
    return [vocab[t] for t in np.argsort(-s)[:k]]

for word in ['king', 'cat', 'man', 'dog']:
    print(f"nearest to {word:5s}:", neighbors(word))
# -> cat: ['kitten','dog','puppy']   king: ['man','boy','queen']

# The analogy direction: king - man + woman
v = En[stoi['king']] - En[stoi['man']] + En[stoi['woman']]
v = v / np.linalg.norm(v)
s = En @ v
for w in ['king', 'man', 'woman']:                    # exclude the inputs
    s[stoi[w]] = -9
print("king - man + woman ->", [vocab[t] for t in np.argsort(-s)[:3]])
# -> ['queen', 'boy', 'prince']

## Visualize the data & results

_Fit GloVe to a global co-occurrence matrix built from a tiny corpus that mixes royalty/people words and animal words — do the learned vectors cluster by topic (and support the king-man+woman analogy) without ever being told the topics?_

### Step 1 — Rebuild the data and refit the model

This cell is self-contained so the plot can run on its own. It repeats the corpus construction and the Adagrad fit from the reference implementation — same seeds, same hyperparameters — so we end up with the identical embeddings `E = W + Wt`.

In [ ]:
import numpy as np

# Rebuild the same two-topic corpus and global co-occurrence matrix
roy = ['king', 'queen', 'prince', 'princess', 'man', 'woman', 'boy', 'girl']
ani = ['cat', 'dog', 'kitten', 'puppy']
rng = np.random.default_rng(42)
sents = [list(rng.choice(roy, size=5, replace=True)) for _ in range(40)]
sents += [list(rng.choice(ani, size=4, replace=True)) for _ in range(30)]

vocab = sorted(set(w for s in sents for w in s))
stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

X = np.zeros((V, V))
for s in sents:
    for a in range(len(s)):
        for b in range(len(s)):
            if a != b:
                X[stoi[s[a]], stoi[s[b]]] += 1.0 / abs(a - b)

# Refit with the same hyperparameters and Adagrad loop
D, xmax, alpha, lr = 16, 30.0, 0.75, 0.05
r = np.random.default_rng(7)
W = r.standard_normal((V, D)) * 0.1
Wt = r.standard_normal((V, D)) * 0.1
b = np.zeros(V)
bt = np.zeros(V)
gW = np.ones_like(W)
gWt = np.ones_like(Wt)
gb = np.ones_like(b)
gbt = np.ones_like(bt)
nz = np.argwhere(X > 0)
fw = lambda x: np.where(x < xmax, (x / xmax) ** alpha, 1.0)

for epoch in range(250):
    r.shuffle(nz)
    for i, j in nz:
        xij = X[i, j]
        res = W[i] @ Wt[j] + b[i] + bt[j] - np.log(xij)
        wt = fw(xij)
        g = 2 * wt * res
        gwi, gwj = g * Wt[j], g * W[i]
        W[i] -= lr / np.sqrt(gW[i]) * gwi
        gW[i] += gwi ** 2
        Wt[j] -= lr / np.sqrt(gWt[j]) * gwj
        gWt[j] += gwj ** 2
        b[i] -= lr / np.sqrt(gb[i]) * g
        gb[i] += g ** 2
        bt[j] -= lr / np.sqrt(gbt[j]) * g
        gbt[j] += g ** 2

E = W + Wt                                     # GloVe sums the two vector sets
print("embeddings shape:", E.shape)

### Step 2 — Project to 2D with PCA and read the coordinates

The embeddings live in 16 dimensions, so we center them and use an SVD (PCA) to project onto the top two principal axes. We scale the result to a readable range and print each word's coordinates.

Look at the numbers: royalty/people words and animal words separate into two groups, confirming GloVe recovered the topic structure from co-occurrence counts alone.

In [ ]:
Ec = E - E.mean(0)                             # center before PCA
U, S, Vt = np.linalg.svd(Ec, full_matrices=False)
P = Ec @ Vt[:2].T                              # project onto the top 2 principal axes
P = P / np.abs(P).max() * 2.0                  # scale to a readable range

for w in vocab:
    x, y = P[stoi[w]]
    print(f"{w:9s} {x:6.3f} {y:6.3f}")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute one term of $J$ for a pair with $X_{ij}=50$, $x_{\max}=100$, $\alpha=0.75$, where $w_i^\top\tilde w_j = 2.5$, $b_i=0.5$, $\tilde b_j=0.5$. (Use $\ln 50 = 3.912023$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Weight: $f(50)=(50/100)^{0.75}=0.5^{0.75}=0.594604$. — _$50\lt x_{\max}$, so use the power-law branch of Eq. 9._
- Residual: $2.5+0.5+0.5-3.912023=-0.412023$. — _Model prediction minus the target $\log X_{ij}$ (Eq. 8 inner term)._
- Square: $(-0.412023)^2=0.169763$. — _Least-squares penalizes the error symmetrically._
- Term: $0.594604\times0.169763=0.100942$. — _Scale the squared error by the weight._

**Answer:** The term is $\approx 0.1009$. Because the model's prediction $3.5$ is already close to $\log 50\approx 3.912$, the residual is small and this pair contributes little to $J$.

</details>

**Problem 2.** Using Table 1's logic, the corpus gives $P(\text{solid}\mid\text{ice})=1.9\times10^{-4}$ and $P(\text{solid}\mid\text{steam})=2.2\times10^{-5}$. Why does GloVe build its model around the ratio of these rather than either probability alone?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ratio $=1.9\times10^{-4} / 2.2\times10^{-5}\approx 8.6$ (paper rounds to 8.9). — _A ratio much greater than 1 flags that 'solid' is specific to 'ice'._
- Compare to an irrelevant probe like 'water': its ratio is $\approx 1.36$, near 1. — _Words related to both (or neither) cancel in the ratio._
- Note both raw probabilities are tiny ($\sim10^{-4}$) and hard to compare directly. — _Raw magnitudes are dominated by overall word frequency, not relatedness._

**Answer:** The ratio cancels out frequency effects and non-discriminative context words (their ratio $\approx 1$), leaving only the signal that distinguishes 'ice' from 'steam'. That is why Eq. 1 sets the model's target to $P_{ik}/P_{jk}$, which after the homomorphism argument becomes the $\log X_{ij}$ regression of Eq. 7.

</details>

**Problem 3.** Ablation: in the CODE, replace the weighting $f(X_{ij})$ with the constant $1$ (plain, unweighted least squares on $\log X$). What should you expect to change, and what should stay the same?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set f to return 1.0 for every count and retrain on the same $X$. — _Removes GloVe's down-weighting of rare and very frequent pairs (Eq. 9 → constant)._
- Watch the rare/noisy pairs (small $X_{ij}$): they now contribute as much as frequent ones. — _Property 2 of $f$ (non-decreasing, small for rare) is what protected against this._
- Re-check the topic clustering and the analogy. — _See whether the qualitative effect survives the change._

**Answer:** With $f\equiv 1$, every nonzero pair counts equally, so a few very frequent pairs and many noisy rare pairs both pull harder on the fit; on a real corpus this degrades the vectors (the paper reports the weighted version helps). On our tiny clean toy corpus the clustering and analogy usually still appear, but the loss landscape is worse-conditioned and the result is more seed-sensitive — illustrating exactly why GloVe introduced $f$.

</details>